# model_experiment_Prophet

მიზანი: Prophet classical/decomposable forecasting approach-ის პატარა ექსპერიმენტი representative Store+Dept series-ზე.

Prophet ყველა Store+Dept series-ზე ცალ-ცალკე დატრენინგდება ძალიან ნელა, ამიტომ აქ მას ვიყენებთ შედარებისთვის და report-ისთვის, არა როგორც მთავარი Kaggle submission მოდელი.

In [1]:
# Run once if packages are missing:
# !pip install -r requirements.txt

In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import pandas as pd
import numpy as np
import joblib
import wandb

from src.config import WANDB_ENTITY, WANDB_PROJECT, MODELS_DIR, SUBMISSIONS_DIR, VALIDATION_FOLDS, VALIDATION_WEEKS, RANDOM_STATE
from src.data_loading import load_raw_data, describe_dataframes, make_submission_frame
from src.validation import evaluate_time_folds, summarize_cv
from src.metrics import wmae
from src.wandb_utils import safe_wandb_init, save_and_log_model, log_dataframe_as_artifact

pd.set_option('display.max_columns', 100)
MODELS_DIR.mkdir(exist_ok=True, parents=True)
SUBMISSIONS_DIR.mkdir(exist_ok=True, parents=True)
print('W&B entity:', WANDB_ENTITY)
print('W&B project:', WANDB_PROJECT)

from src.classical import evaluate_prophet_selected_series, top_series

W&B entity: gchal22-free-university-of-tbilisi-
W&B project: store_sales_forecast


In [3]:
wandb.login()
data = load_raw_data()
train = data['train'].copy()
train['Date'] = pd.to_datetime(train['Date'])

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\West\_netrc.
wandb: Currently logged in as: gchal22 (gchal22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
N_SERIES = 5
series_keys = top_series(train, n=N_SERIES)
series_keys

[(14, 92), (2, 92), (20, 92), (13, 92), (4, 92)]

In [5]:
with safe_wandb_init(
    'Prophet_Selected_Series',
    'Classical',
    'selected_series_experiment',
    config={'n_series': N_SERIES, 'yearly_seasonality': True, 'country_holidays': 'US'},
) as run:
    prophet_results = evaluate_prophet_selected_series(train, series_keys=series_keys, validation_weeks=8)
    wandb.log({'prophet_results': wandb.Table(dataframe=prophet_results)})
    if prophet_results['wmae'].notna().any():
        wandb.log({'prophet_selected_series_mean_wmae': float(prophet_results['wmae'].mean())})
    log_dataframe_as_artifact(run, prophet_results, 'prophet-selected-series-results', 'validation_results', 'reports/prophet_selected_series_results.csv')

prophet_results

Matplotlib is building the font cache; this may take a moment.
Importing plotly failed. Interactive plots will not work.
21:37:47 - cmdstanpy - INFO - Chain [1] start processing
21:37:48 - cmdstanpy - INFO - Chain [1] done processing
21:37:48 - cmdstanpy - INFO - Chain [1] start processing
21:37:48 - cmdstanpy - INFO - Chain [1] done processing
21:37:48 - cmdstanpy - INFO - Chain [1] start processing
21:37:48 - cmdstanpy - INFO - Chain [1] done processing
21:37:49 - cmdstanpy - INFO - Chain [1] start processing
21:37:49 - cmdstanpy - INFO - Chain [1] done processing
21:37:49 - cmdstanpy - INFO - Chain [1] start processing
21:37:49 - cmdstanpy - INFO - Chain [1] done processing


prophet_selected_series_mean_wmae,▁
prophet_selected_series_mean_wmae,10977.26594


,store,dept,method,wmae,status
0,14,92,Prophet,18924.032950,ok
1,2,92,Prophet,12433.891140,ok
2,20,92,Prophet,10405.894865,ok
3,13,92,Prophet,7596.342830,ok
4,4,92,Prophet,5526.167914,ok


## Report note

Prophet-ის შედეგები შეადარე seasonal naive-სა და tree-based CV-ს. თუ Prophet selected series-ზე კარგადაც მუშაობს, მთელ dataset-ზე მისი ცალ-ცალკე fit-ება მაინც ძვირია და deployment/inference უფრო რთული ხდება.